In [ ]:
import os
import boto3
import botocore
import json


In [ ]:
aws_region = "us-east-1"

# Set these via environment variables before running notebooks
guardrail_id = os.environ.get("BEDROCK_GUARDRAIL_ID")
logging_role_arn = os.environ.get("BEDROCK_LOGGING_ROLE_ARN")
guardrail_version = os.environ.get("BEDROCK_GUARDRAIL_VERSION", "DRAFT")

# Created the Bedrock client
bedrock_client = boto3.client(
    service_name="bedrock",
    region_name=aws_region,
)

# Created the Bedrock runtime client
bedrock_runtime_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=aws_region
    )

# Print the URL for the Bedrock client
print(bedrock_client.meta.endpoint_url)
print(bedrock_runtime_client.meta.endpoint_url)


## Grounding Policy
It has two main component Grounding and Relevance

### Data for the RAG

In [ ]:

SOURCE_DOCUMENT = """
NovaTech Solutions – Company Overview (Internal Document)
==========================================================

Company Name   : NovaTech Solutions
Founded        : 2019
Headquarters   : Austin, Texas, USA
Employees      : 245 (as of September 2024)
CEO            : Jennifer Walsh
Primary Market : Healthcare sector
Main Product   : CloudSync – a secure file synchronization platform
                 for healthcare providers.
Latest Update  : CloudSync v3.2 was released in August 2024,
                 adding HIPAA-compliant audit logs.

Mission: To simplify data management for healthcare organisations
while maintaining the highest standards of security and compliance.
"""

GROUND_QUESTION = [
    "What is the main product of NovaTech Solutions?",
    "Who is the CEO of NovaTech Solutions?",
    "What is the mission of NovaTech Solutions?",
    "What is the headquarters of NovaTech Solutions?",
]

NON_GROUND_QUESTION = [
    "What is the revenue of NovaTech Solutions in 2000?",
    "What is CEO wife's Share in the company?",
]

### Implementing the RAG Def 


In [ ]:
def executeRAG(question, guardRailsId, guardRailsVersion="DRAFT"):

    prompt = f"""
You are a RAG-based question answering assistant.

SOURCE DOCUMENT:
----------------
{SOURCE_DOCUMENT}
----------------

QUESTION:
{question}

ANSWER:
"""

    response = bedrock_runtime_client.converse(

        # Model
        modelId="amazon.nova-micro-v1:0",

        # User message
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "text": prompt
                    }
                ]
            }
        ],

        # Model configuration
        inferenceConfig={
            "maxTokens": 500,
            "temperature": 0.0,
            "topP": 0.9
        },

        # Guardrail configuration
        guardrailConfig={
            "guardrailIdentifier": guardRailsId,
            "guardrailVersion": guardRailsVersion,
            "trace": "enabled"
        }
    )

    return {
        "question": question,

        "answer": response["output"]["message"]["content"][0]["text"],

        "usage": response.get("usage", {}),

        "stopReason": response.get("stopReason"),

        "guardrailTrace": response.get("trace")
    }

## Adding the GuardRail Filter


In [ ]:
response = bedrock_client.create_guardrail(
    name="RAG-Grounding-Guardrail",

    description=(
        "Guardrail for RAG applications that checks whether "
        "model responses are grounded in the provided source "
        "documents and relevant to the user's query."
    ),

    blockedInputMessaging=(
        "The request could not be processed because it does "
        "not meet the grounding requirements."
    ),

    blockedOutputsMessaging=(
        "The response was blocked because it could not be "
        "sufficiently grounded in the provided source."
    ),

    contextualGroundingPolicyConfig={
        "filtersConfig": [

            # Check whether the response is supported
            # by the source document
            {
                "type": "GROUNDING",
                "threshold": 0.75,
                "action": "BLOCK",
                "enabled": True
            },

            # Check whether the response is relevant
            # to the user's question
            {
                "type": "RELEVANCE",
                "threshold": 0.75,
                "action": "BLOCK",
                "enabled": True
            }
        ]
    }
)

print(response)

## Using the Rag

In [ ]:
result = executeRAG("What is total Revenue of NovaTech Solutions in 2000 ?",guardrail_id)
print("=" * 60)
print("RAG RESULT")
print("=" * 60)
print(f"Question     : {result['question']}")
print(f"Answer       : {result['answer']}")
print("-" * 60)
print(f"Input tokens : {result['usage'].get('inputTokens')}")
print(f"Output tokens: {result['usage'].get('outputTokens')}")
print(f"Stop reason  : {result['stopReason']}")
print("=" * 60)
